In [1]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

# Load preprocessed binary dataset
X_train = pd.read_csv('../../dataset_processed/X_train_final.csv')
y_train = pd.read_csv('../../dataset_processed/y_train_final.csv').values.ravel()
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

In [2]:
# Define 5-Fold Stratified CV strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize Baseline Model with SMOTE (Option A: 30% synthetic minority over-sampling)
smote = SMOTE(sampling_strategy=0.30, random_state=42)
baseline_clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
baseline_pipeline = ImbPipeline([('smote', smote), ('classifier', baseline_clf)])

# Run cross-validation
scoring = ['f1_macro', 'recall_macro', 'accuracy']
cv_results = cross_validate(baseline_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print(f"Mean CV Macro F1: {cv_results['test_f1_macro'].mean():.4f}")
print(f"Mean CV Macro Recall: {cv_results['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy: {cv_results['test_accuracy'].mean():.4f}")

# Fit on full data for classification report
baseline_pipeline.fit(X_train, y_train)
y_pred = baseline_pipeline.predict(X_train)
report = classification_report(y_train, y_pred)
print("\nClassification Report on Full Train Data:")
print(report)

# Save metrics
os.makedirs('results', exist_ok=True)
with open('results/baseline_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {cv_results['test_f1_macro'].mean():.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data:\n")
    f.write(report)

Mean CV Macro F1: 0.5124
Mean CV Macro Recall: 0.6096
Mean CV Accuracy: 0.6363

Classification Report on Full Train Data:
              precision    recall  f1-score   support

           0       0.92      0.64      0.76     69327
           1       0.17      0.58      0.27      9004

    accuracy                           0.64     78331
   macro avg       0.55      0.61      0.51     78331
weighted avg       0.84      0.64      0.70     78331



## Phase 2: Hyperparameter Tuning with Optuna

In [3]:
import warnings
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

def objective(trial):
    # Hyperparameter search space
    solver = trial.suggest_categorical('solver', ['saga', 'lbfgs'])
    if solver == 'saga':
        penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
        if penalty == 'elasticnet':
            l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.9)
        else:
            l1_ratio = None
    else:
        penalty = 'l2'
        l1_ratio = None
        
    C = trial.suggest_float('C', 1e-4, 50.0, log=True)
    tol = trial.suggest_float('tol', 1e-4, 1e-2, log=True)
    
    model = LogisticRegression(
        class_weight='balanced',
        max_iter=500,
        random_state=42,
        C=C,
        solver=solver,
        penalty=penalty,
        l1_ratio=l1_ratio,
        tol=tol
    )
    
    pipeline = ImbPipeline([
        ('smote', SMOTE(sampling_strategy=0.30, random_state=42)),
        ('classifier', model)
    ])
    
    res = cross_validate(pipeline, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    return res['test_score'].mean()

# Set optuna logging to warning only
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
print("Starting Logistic Regression Deep Hyperparameter Tuning (30 trials)...")
study.optimize(objective, n_trials=30, show_progress_bar=False)

print(f"\nBest Params: {study.best_params}")
print(f"Best Macro F1: {study.best_value:.4f}")

Starting Logistic Regression Deep Hyperparameter Tuning (30 trials)...

Best Params: {'solver': 'saga', 'penalty': 'elasticnet', 'l1_ratio': 0.4255, 'C': 0.00045, 'tol': 0.00013}
Best Macro F1: 0.5207


## Phase 3: Tuned Model Evaluation

In [4]:
# Initialize the tuned LogisticRegression using the best parameters
best_params = study.best_params.copy()
solver = best_params.get('solver', 'saga')
penalty = best_params.get('penalty', 'l2' if solver == 'lbfgs' else 'elasticnet')
l1_ratio = best_params.get('l1_ratio', None)
C = best_params['C']
tol = best_params.get('tol', 1e-4)

tuned_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    solver=solver,
    penalty=penalty,
    l1_ratio=l1_ratio,
    C=C,
    tol=tol
)

tuned_pipeline = ImbPipeline([
    ('smote', SMOTE(sampling_strategy=0.30, random_state=42)),
    ('classifier', tuned_model)
])

# Run cross-validation
cv_results_tuned = cross_validate(tuned_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

mean_f1_tuned = cv_results_tuned['test_f1_macro'].mean()
print(f"Mean CV Macro F1 (Tuned): {mean_f1_tuned:.4f}")
print(f"Mean CV Macro Recall (Tuned): {cv_results_tuned['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy (Tuned): {cv_results_tuned['test_accuracy'].mean():.4f}")

# Fit on full training data
tuned_pipeline.fit(X_train, y_train)
y_pred_tuned = tuned_pipeline.predict(X_train)
report_tuned = classification_report(y_train, y_pred_tuned)
print("\nClassification Report on Full Train Data (Tuned):")
print(report_tuned)

# Save metrics
with open('results/tuned_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {mean_f1_tuned:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_tuned['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_tuned['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Tuned):\n")
    f.write(report_tuned)

Mean CV Macro F1 (Tuned): 0.5207
Mean CV Macro Recall (Tuned): 0.5931
Mean CV Accuracy (Tuned): 0.6673

Classification Report on Full Train Data (Tuned):
              precision    recall  f1-score   support

           0       0.91      0.67      0.77     69327
           1       0.17      0.52      0.26      9004

    accuracy                           0.65     78331
   macro avg       0.54      0.60      0.52     78331
weighted avg       0.83      0.65      0.71     78331



## Phase 4: Model Comparison

In [5]:
# Baseline vs Tuned Comparison
baseline_f1 = cv_results['test_f1_macro'].mean()
print("=== Logistic Regression Performance ===")
print(f"Baseline Macro F1: {baseline_f1:.4f}")
print(f"Tuned Macro F1:    {mean_f1_tuned:.4f}")
print(f"Improvement:       {mean_f1_tuned - baseline_f1:+.4f}")

=== Logistic Regression Performance ===
Baseline Macro F1: 0.5124
Tuned Macro F1:    0.5207
Improvement:       +0.0083


## Phase 5: Save Final Model

In [6]:
import joblib

# Persist the tuned model with best hyperparameters
joblib.dump(tuned_pipeline, 'logistic_regression_best_model.pkl')
print("Model saved successfully to logistic_regression_best_model.pkl")

Model saved successfully to logistic_regression_best_model.pkl
